# Yug — quickstart

Forecast a time series in a few lines, with no training and no feature engineering.

Yug is a **zero-shot probabilistic** forecaster: you hand it history, it returns
a set of quantile levels over the horizon you ask for. It has never seen your series.

---

**Install**

```bash
pip install yug matplotlib
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from yug import YugPipeline

# A Hugging Face Hub id, or a path to a local checkpoint directory.
MODEL_ID = "birlaailabs/yug"

pipeline = YugPipeline.from_pretrained(MODEL_ID, device_map="cpu")
pipeline

## 1. A series to forecast

Daily data with a weekly cycle, a slow annual cycle, a mild upward trend and noise —
the shape of a lot of real demand data. We hold out the last 96 points as truth.

In [ ]:
HORIZON = 64

rng = np.random.default_rng(15)
t = np.arange(1000)

signal = (
    100.0  # level
    + 0.05 * t  # trend
    + 12.0 * np.sin(2 * np.pi * t / 7)  # weekly cycle
    + 25.0 * np.sin(2 * np.pi * t / 365)  # annual cycle
    + rng.normal(0, 3.0, len(t))  # noise
).astype(np.float32)

context, truth = signal[:-HORIZON], signal[-HORIZON:]
print(f"{len(context)} points of history, forecasting {HORIZON} ahead")

## 2. Forecast

`freq` is a real input to the model, not bookkeeping — it selects a learned embedding,
so passing the wrong one degrades the forecast.

In [ ]:
forecast = pipeline.predict(
    context,
    prediction_length=HORIZON,
    freq="D",
    num_samples=100,  # sampled trajectories behind the band
    seed=0,  # makes the call reproducible
)

forecast

In [ ]:
median = forecast.median
band = forecast.interval()  # widest trained pair, here 0.1 / 0.9

print("median :", np.round(median[:6], 2))
print("lower  :", np.round(band["lower"][:6], 2))
print("upper  :", np.round(band["upper"][:6], 2))

## 3. Plot it

In [ ]:
future_x = np.arange(len(context), len(context) + HORIZON)

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(
    np.arange(len(context))[-250:],
    context[-250:],
    color="#3a3a3a",
    lw=1.1,
    label="history",
)
ax.plot(future_x, truth, color="#1a7f5a", lw=1.4, label="actual")
ax.plot(future_x, median, color="#c2410c", lw=1.6, label="forecast (median)")
ax.fill_between(
    future_x,
    band["lower"],
    band["upper"],
    color="#c2410c",
    alpha=0.18,
    label="80% interval",
)
ax.axvline(len(context), color="#999", ls="--", lw=1)
ax.set_xlabel("step")
ax.set_ylabel("value")
ax.set_title("Yug zero-shot forecast")
ax.legend(loc="upper left", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Score it

Two numbers worth looking at together:

- **MASE** — error relative to a seasonal-naive baseline. Below 1.0 beats it.
- **Coverage** — how often the truth actually falls inside the 80% band. It should be
  near 0.8. A sharp median with badly calibrated intervals is a worse model than the
  headline error suggests.

In [ ]:
season = 7

naive_error = np.mean(np.abs(context[season:] - context[:-season]))
mase = np.mean(np.abs(median - truth)) / naive_error
coverage = np.mean((truth >= band["lower"]) & (truth <= band["upper"]))

print(f"MAE       {np.mean(np.abs(median - truth)):8.3f}")
print(f"MASE      {mase:8.3f}   (< 1.0 beats seasonal naive)")
print(f"Coverage  {coverage:8.3f}   (nominal 0.80)")

## 5. Many series at once

Pass a list of arrays. They may have different lengths and different frequencies.

In [ ]:
series_a = context
series_b = context[:400] * 0.5 + 40
series_c = np.cumsum(rng.normal(0, 1, 600)).astype(np.float32) + 200

batch = pipeline.predict(
    [series_a, series_b, series_c],
    prediction_length=48,
    freq=["D", "D", "H"],
    num_samples=100,
    item_ids=["a", "b", "c"],
    seed=0,
)

print(batch.values.shape, " (series, quantiles, horizon)")
batch.to_dataframe().head()

### Covariates

Pass a 2-D array `(n_channels, T)` inside a list to condition a single forecast on
covariates. Row 0 is the target; the remaining rows are aligned historical signals.

```python
multivariate = np.stack([target, covariate_1, covariate_2])
pipeline.predict([multivariate], prediction_length=48, freq="H")
```

### A worked synthetic example: a shared daily pattern

Start with a gradual upward trend and a smooth daily cycle. The target and two
related measurements share this pattern, with independent measurement noise.
The related measurements are less noisy, giving the model extra observations of
the underlying cycle. We forecast two complete days so the peaks and troughs are
visible, using only historical observations from all three channels.

This reproducible **synthetic illustration** was selected to show a favorable use
of related signals. It is not a benchmark or a guarantee that covariates help every
series. The modest error difference can change with the sampling seed.
Both calls use the same 2,048-point context, horizon, frequency, sample
count, and forecast seed. Longer contexts can be explored by changing
`SENSOR_CONTEXT_LENGTH`; 8,192 points exceeds Yug's 2,048-point training context.

For horizons beyond the model's 64-point output block, covariate rollout currently
tiles the sampled target block across all channels, so longer forecasts need care.


In [ ]:
# Hourly observations with a modest trend and a daily cycle.
SENSOR_HISTORY = 8192
SENSOR_HORIZON = 48
SENSOR_CONTEXT_LENGTH = 2048

sensor_rng = np.random.default_rng(42)
sensor_t = np.arange(SENSOR_HISTORY + SENSOR_HORIZON, dtype=np.float64)
shared_signal = (
    100.0
    + 0.005 * sensor_t
    + 12.0 * np.sin(2 * np.pi * sensor_t / 24)
)
sensor_1 = shared_signal + sensor_rng.normal(0, 0.1, len(sensor_t))
sensor_2 = shared_signal + sensor_rng.normal(0, 0.1, len(sensor_t))
response = shared_signal + sensor_rng.normal(0, 1.0, len(sensor_t))

sensor_panel = np.stack([response, sensor_1, sensor_2]).astype(np.float32)
sensor_context = sensor_panel[:, :SENSOR_HISTORY]
sensor_truth = sensor_panel[0, SENSOR_HISTORY:]

fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(sensor_t[:SENSOR_HISTORY], sensor_context[0], color="#3a3a3a", lw=0.7)
ax.set(title="Synthetic target: gradual trend + daily seasonality",
       xlabel="hour", ylabel="value")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
settings = dict(
    prediction_length=SENSOR_HORIZON,
    context_length=SENSOR_CONTEXT_LENGTH,
    freq="H",
    num_samples=30,
    seed=0,
)
sensor_forecasts = {
    "Target only": pipeline.predict(sensor_context[0], **settings),
    "With related measurements": pipeline.predict([sensor_context], **settings),
}


In [ ]:
sensor_future_x = np.arange(SENSOR_HORIZON)
sensor_history_x = np.arange(-48, 0)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True, sharey=True)

for ax, (name, fc) in zip(axes, sensor_forecasts.items(), strict=True):
    band = fc.interval()
    ax.plot(sensor_history_x, sensor_context[0, -48:], color="#3a3a3a",
            lw=1.0, label="target history")
    ax.plot(sensor_future_x, sensor_truth, color="#1a7f5a", lw=1.4,
            label="actual")
    ax.plot(sensor_future_x, fc.median, color="#c2410c", lw=1.6,
            label="forecast median")
    ax.fill_between(sensor_future_x, band["lower"], band["upper"],
                    color="#c2410c", alpha=0.18, label="80% interval")
    ax.axvline(-0.5, color="#999", ls="--", lw=1)
    mae = np.mean(np.abs(fc.median - sensor_truth))
    ax.set_title(f"{name} | MAE {mae:.2f}")
    ax.set_xlabel("hours from forecast start")
    ax.set_ylabel("value")
    ax.legend(loc="lower left", frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Yug — two days ahead on a synthetic daily cycle", fontsize=13)
plt.tight_layout()
plt.show()

print(f"{'Method':26s}  {'MAE':>7}  {'Coverage':>8}")
for name, fc in sensor_forecasts.items():
    band = fc.interval()
    mae = np.mean(np.abs(fc.median - sensor_truth))
    coverage = np.mean(
        (sensor_truth >= band["lower"]) & (sensor_truth <= band["upper"])
    )
    print(f"{name:26s}  {mae:7.2f}  {coverage:8.2f}")
uni_mae = np.mean(np.abs(sensor_forecasts["Target only"].median - sensor_truth))
multi_mae = np.mean(
    np.abs(sensor_forecasts["With related measurements"].median - sensor_truth)
)
print(f"MAE reduction with related measurements: {1 - multi_mae / uni_mae:.1%}")
